In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, count, when, isnan, countDistinct
from pyspark.sql import SparkSession

jar_paths = [
    "/home/jovyan/work/jars/delta-spark_2.12-3.1.0.jar",
    "/home/jovyan/work/jars/delta-storage-3.1.0.jar",
    "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar",
    "/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar"
]
jars_string = ",".join(jar_paths)

spark = SparkSession.builder \
    .appName("LakehouseSetup_Offline") \
    .config("spark.jars", jars_string) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark Version: {spark.version}")
print("Connected successfully in Offline Mode! Ready to build the Lakehouse.")

Spark Version: 3.5.0
Connected successfully in Offline Mode! Ready to build the Lakehouse.


In [10]:
import pandas as pd
from pyspark.sql.functions import col, count, when


# Danh sách 9 bảng raw
bronze_files = {
    "Orders": "s3a://olist-data/bronze/olist_orders_dataset.csv",
    "Reviews": "s3a://olist-data/bronze/olist_order_reviews_dataset.csv",
    "Items": "s3a://olist-data/bronze/olist_order_items_dataset.csv",
    "Products": "s3a://olist-data/bronze/olist_products_dataset.csv",
    "Payments": "s3a://olist-data/bronze/olist_order_payments_dataset.csv",
    "Customers": "s3a://olist-data/bronze/olist_customers_dataset.csv",
    "Sellers": "s3a://olist-data/bronze/olist_sellers_dataset.csv",
    "Geolocation": "s3a://olist-data/bronze/olist_geolocation_dataset.csv",
    "Translation": "s3a://olist-data/bronze/product_category_name_translation.csv"
}

def advanced_bronze_eda(table_name, file_path):
    print(f"{'='*50}")
    print(f"PROFILING Table: {table_name}")
    print(f"{'='*50}")
    
    df = spark.read.csv(file_path, header=True, inferSchema=True)
    total_rows = df.count()
    total_cols = len(df.columns)
    print(f"Shape: {total_rows:,} rows | {total_cols} cols")
    
    unique_rows = df.dropDuplicates().count()
    duplicate_rows = total_rows - unique_rows
    dup_pct = (duplicate_rows / total_rows) * 100 if total_rows > 0 else 0
    
    if duplicate_rows > 0:
        print(f"There are {duplicate_rows:,} duplicate row : ({dup_pct:.2f}%)")
    else:
        print(f"-> No Duplicate row")
    
    null_exprs = []
    for c in df.columns:
        null_exprs.append(
            (count(when(col(c).isNull(), c)) / total_rows * 100).alias(c)
        )
    null_df = df.select(null_exprs).toPandas().T
    null_df.columns = ["NULL (%)"]
    null_df_filtered = null_df[null_df["NULL (%)"] > 0].round(2)
    
    if null_df_filtered.empty:
        print("-> No NULL detected.")
    else:
        print(null_df_filtered.to_string())
    print("+"*100)
    print("\n")

# Thực thi vòng lặp chạy toàn bộ 9 bảng
for name, path in bronze_files.items():
    advanced_bronze_eda(name, path)

print("Finish EDA in Bronze layer")

PROFILING Table: Orders
Shape: 99,441 rows | 8 cols
-> No Duplicate row
                               NULL (%)
order_approved_at                  0.16
order_delivered_carrier_date       1.79
order_delivered_customer_date      2.98
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


PROFILING Table: Reviews
Shape: 104,162 rows | 7 cols
There are 85 duplicate row : (0.08%)
                         NULL (%)
review_id                    0.00
order_id                     2.15
review_score                 2.28
review_comment_title        88.47
review_comment_message      60.56
review_creation_date         8.41
review_answer_timestamp      8.43
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


PROFILING Table: Items
Shape: 112,650 rows | 7 cols
-> No Duplicate row
-> No NULL detected.
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


PRO